In [1]:
import sys
import pathlib
import pandas as pd
import numpy as np

# Adjusting paths to find the 'data' directory relative to the notebook location
# This replaces the __file__ logic used in scripts
DATA_DIR = pathlib.Path.cwd().parent / "data"
raw_path = DATA_DIR / "raw_sample.parquet"

if not raw_path.exists():
    print(f"❌ No cached data found at {raw_path}")
    print("Run the Streamlit app first to fetch data, or run data/fetch.py directly.")
else:
    df = pd.read_parquet(raw_path)
    print(f"✅ Data loaded successfully. Shape: {df.shape}")


✅ Data loaded successfully. Shape: (5000, 19)


In [2]:
# Basic shape & memory
info_df = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Memory Usage"],
    "Value": [
        f"{df.shape[0]:,}",
        df.shape[1],
        f"{df.memory_usage(deep=True).sum() / 1e6:.2f} MB"
    ]
})
info_df

,Metric,Value
0,Rows,"5,000"
1,Columns,19
2,Memory Usage,1.61 MB


In [3]:
# Columns, Dtypes, and Null Analysis
schema_summary = pd.DataFrame({
    "Dtype": df.dtypes,
    "Null Count": df.isnull().sum(),
    "Null %": (df.isnull().sum() / len(df) * 100).round(2)
})
schema_summary.index.name = "Column"
schema_summary.style.background_gradient(subset=["Null %"], cmap="Reds")

,Dtype,Null Count,Null %
Column,,,
VendorID,int64,0,0.000000
tpep_pickup_datetime,object,0,0.000000
tpep_dropoff_datetime,object,0,0.000000
passenger_count,float64,0,0.000000
trip_distance,float64,0,0.000000
RatecodeID,float64,0,0.000000
store_and_fwd_flag,object,0,0.000000
PULocationID,int64,0,0.000000
DOLocationID,int64,0,0.000000


In [4]:
# Numeric Summary
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols].describe().round(2)

Column,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,1.75,1.56,3.27,1.16,157.38,156.31,1.23,17.92,1.55,0.49,3.38,0.20,0.98,26.25,2.29,0.03
std,0.43,0.99,3.40,3.13,70.27,75.04,0.52,15.16,1.13,0.11,3.79,1.33,0.20,18.05,0.76,0.18
min,1.00,0.00,0.00,1.00,4.00,1.00,1.00,-80.00,-1.00,-0.50,0.00,-15.75,-1.00,-96.75,-2.50,-1.25
25%,2.00,1.00,1.24,1.00,107.00,90.00,1.00,9.30,1.00,0.50,0.84,0.00,1.00,16.27,2.50,0.00
50%,2.00,1.00,2.24,1.00,151.00,151.00,1.00,14.20,1.00,0.50,2.86,0.00,1.00,21.90,2.50,0.00
75%,2.00,2.00,4.10,1.00,233.00,233.00,1.00,22.60,1.00,0.50,4.68,0.00,1.00,31.44,2.50,0.00
max,2.00,6.00,40.20,99.00,265.00,265.00,4.00,243.00,8.50,0.50,75.00,19.65,1.00,261.50,2.50,1.25


In [5]:
# Top 5 values for categorical columns
obj_cols = df.select_dtypes(include=["object", "category"]).columns

for col in obj_cols:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    display(df[col].value_counts().head(5).to_frame())

--- tpep_pickup_datetime (4246 unique) ---


,count
tpep_pickup_datetime,
2023-01-01T00:39:27.000,5
2023-01-01T03:16:54.000,5
2023-01-01T02:17:07.000,4
2023-01-01T00:33:15.000,4
2023-01-01T00:44:06.000,4


--- tpep_dropoff_datetime (4281 unique) ---


,count
tpep_dropoff_datetime,
2023-01-01T00:55:24.000,5
2023-01-01T02:49:17.000,4
2023-01-01T01:12:38.000,4
2023-01-01T01:00:58.000,4
2023-01-01T03:05:33.000,4


--- store_and_fwd_flag (2 unique) ---


,count
store_and_fwd_flag,
N,4960
Y,40


In [6]:
# Datetime Column Coverage
dt_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns

if not dt_cols.empty:
    dt_summary = df[dt_cols].agg(['min', 'max']).T
    display(dt_summary)
else:
    print("No datetime columns found.")

No datetime columns found.


In [7]:
# Final Inspection
df.head()

Column,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2023-01-01T02:12:51.000,2023-01-01T02:17:31.000,2.0,0.80,1.0,N,90,107,1,6.5,3.5,0.5,2.30,0.00,1.0,13.80,2.5,0.00
1,2,2023-01-01T00:47:39.000,2023-01-01T01:23:27.000,1.0,20.19,2.0,N,132,239,1,70.0,0.0,0.5,8.00,6.55,1.0,89.80,2.5,1.25
2,2,2023-01-01T01:01:54.000,2023-01-01T01:35:58.000,1.0,4.57,1.0,N,125,239,1,32.4,1.0,0.5,7.48,0.00,1.0,44.88,2.5,0.00
3,2,2023-01-01T00:22:40.000,2023-01-01T00:29:42.000,2.0,0.93,1.0,N,237,141,2,7.9,1.0,0.5,0.00,0.00,1.0,12.90,2.5,0.00
4,2,2023-01-01T02:50:53.000,2023-01-01T03:02:37.000,1.0,7.56,1.0,N,132,63,2,29.6,1.0,0.5,0.00,0.00,1.0,33.35,0.0,1.25
